# Financial News Sentiment Analysis with FinBERT

This notebook builds an end-to-end, production-ready sentiment classifier for financial news using the attached dataset. We will explore the data, clean the text, fine-tune FinBERT, evaluate results, improve F1-score with class balancing and tuning, and save a ready-to-use model.

**Dataset**: `all-data.csv` (label, text)

## 1. Import Libraries 



In [ ]:
import importlib
import os
import random
import re
import sys
import subprocess
from dataclasses import dataclass

# Install missing packages (safe to re-run)
required_packages = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "transformers",
    "torch"
]

for pkg in required_packages:
    try:
        importlib.import_module(pkg if pkg != "scikit-learn" else "sklearn")
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Runtime device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 2. Load Dataset from CSV

We load the CSV into a DataFrame. The file has two columns: `label` and `text`.

In [ ]:
data_path = "all-data.csv"

# The dataset is label, text without a header
raw_df = pd.read_csv(
    data_path,
    header=None,
    names=["label", "text"],
    encoding="utf-8",
    quotechar='"',
    escapechar="\\",
)

# Fallback in case extra columns appear
if raw_df.shape[1] > 2:
    extra_cols = raw_df.columns[2:]
    raw_df["text"] = raw_df["text"].astype(str) + "," + raw_df[extra_cols].astype(str).agg(",".join, axis=1
    )
    raw_df = raw_df[["label", "text"]]

raw_df.head()

## 3. Initial Data Inspection & Schema Checks

We check basic schema, missing values, duplicates, and label distribution to ensure data integrity.

In [ ]:
df = raw_df.copy()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Label distribution:\n", df["label"].value_counts())

## 4. EDA: Class Balance & Text Length Distributions

We visualize class balance and text length distribution to understand the dataset structure.

In [ ]:
df["text_length"] = df["text"].astype(str).apply(len)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
plt.title("Sentiment Class Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.histplot(df["text_length"], bins=30, kde=True)
plt.title("Text Length Distribution")
plt.xlabel("Number of Characters")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="label", y="text_length")
plt.title("Text Length by Label")
plt.xlabel("Label")
plt.ylabel("Number of Characters")
plt.tight_layout()
plt.show()

## 5. Text Cleaning & Normalization Pipeline

We normalize text by lowercasing, removing URLs and punctuation, and collapsing extra whitespace.

In [ ]:
url_pattern = re.compile(r"https?://\S+|www\.\S+")
punct_pattern = re.compile(r"[^a-z0-9\s]")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = url_pattern.sub(" ", text)
    text = punct_pattern.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["text"].apply(clean_text)
df[["label", "text", "clean_text"]].head()

## 6. Label Encoding & Dataset Split (Train/Val/Test)

We encode sentiment labels and create stratified train, validation, and test splits.

In [ ]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df = df[df["label"].isin(label_map)].copy()
df["label_id"] = df["label"].map(label_map)

X = df["clean_text"].tolist()
y = df["label_id"].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

## 7. Tokenization with FinBERT

We load FinBERT's tokenizer, tokenize the texts, and prepare PyTorch datasets for training.

In [ ]:
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts):
    return tokenizer(
        texts,
        truncation=True,
        padding=False,
        max_length=128,
    )

train_encodings = tokenize_texts(X_train)
val_encodings = tokenize_texts(X_val)
test_encodings = tokenize_texts(X_test)


class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = NewsDataset(train_encodings, y_train)
val_dataset = NewsDataset(val_encodings, y_val)
test_dataset = NewsDataset(test_encodings, y_test)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_dataset[0].keys()

## 8. Baseline FinBERT Fine-Tuning

We fine-tune FinBERT using a baseline configuration to establish a starting point.

In [ ]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v: k for k, v in id2label.items()}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to(DEVICE)

baseline_args = TrainingArguments(
    output_dir="finbert-baseline",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    seed=SEED,
)

baseline_trainer = Trainer(
    model=baseline_model,
    args=baseline_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

baseline_trainer.train()

## 9. Evaluation Metrics & Confusion Matrix

We evaluate the baseline model on the test set and visualize the confusion matrix.

In [ ]:
baseline_pred = baseline_trainer.predict(test_dataset)
baseline_logits = baseline_pred.predictions
baseline_labels = baseline_pred.label_ids
baseline_preds = np.argmax(baseline_logits, axis=1)

baseline_precision, baseline_recall, baseline_f1, _ = precision_recall_fscore_support(
    baseline_labels, baseline_preds, average="weighted", zero_division=0
)
baseline_accuracy = accuracy_score(baseline_labels, baseline_preds)

baseline_metrics = {
    "accuracy": baseline_accuracy,
    "precision": baseline_precision,
    "recall": baseline_recall,
    "f1": baseline_f1,
}
baseline_metrics

In [ ]:
cm = confusion_matrix(baseline_labels, baseline_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[id2label[i] for i in range(3)],
    yticklabels=[id2label[i] for i in range(3)],
)
plt.title("Baseline Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 10. F1 Improvement: Class Balancing & Hyperparameter Tuning

We apply class weighting and tune key hyperparameters to improve F1-score.

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=np.array(y_train),
)


class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.model.device)

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


tuning_configs = [
    {"lr": 1e-5, "batch": 16, "epochs": 3},
    {"lr": 2e-5, "batch": 32, "epochs": 3},
]

best_f1 = -1
best_trainer = None
best_config = None

for idx, cfg in enumerate(tuning_configs, start=1):
    tuned_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
    ).to(DEVICE)

    tuned_args = TrainingArguments(
        output_dir=f"finbert-tuned-{idx}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["batch"],
        per_device_eval_batch_size=cfg["batch"],
        num_train_epochs=cfg["epochs"],
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
        seed=SEED,
    )

    trainer = WeightedTrainer(
        model=tuned_model,
        args=tuned_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    trainer.train()
    eval_metrics = trainer.evaluate(val_dataset)

    if eval_metrics["eval_f1"] > best_f1:
        best_f1 = eval_metrics["eval_f1"]
        best_trainer = trainer
        best_config = cfg

best_config, best_f1

In [ ]:
tuned_pred = best_trainer.predict(test_dataset)
tuned_logits = tuned_pred.predictions
tuned_labels = tuned_pred.label_ids
tuned_preds = np.argmax(tuned_logits, axis=1)

tuned_precision, tuned_recall, tuned_f1, _ = precision_recall_fscore_support(
    tuned_labels, tuned_preds, average="weighted", zero_division=0
)
tuned_accuracy = accuracy_score(tuned_labels, tuned_preds)

tuned_metrics = {
    "accuracy": tuned_accuracy,
    "precision": tuned_precision,
    "recall": tuned_recall,
    "f1": tuned_f1,
}
tuned_metrics

In [ ]:
tuned_cm = confusion_matrix(tuned_labels, tuned_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(
    tuned_cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=[id2label[i] for i in range(3)],
    yticklabels=[id2label[i] for i in range(3)],
)
plt.title("Tuned Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 11. Results Visualization & Model Comparison

We compare baseline and tuned model metrics side by side.

In [ ]:
comparison_df = pd.DataFrame([
    {"model": "baseline", **baseline_metrics},
    {"model": "tuned", **tuned_metrics},
])

comparison_df

In [ ]:
plt.figure(figsize=(7, 4))
metrics_to_plot = ["accuracy", "precision", "recall", "f1"]

for metric in metrics_to_plot:
    plt.plot(
        comparison_df["model"],
        comparison_df[metric],
        marker="o",
        label=metric,
    )

plt.title("Baseline vs Tuned Metrics")
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

## 12. Save Trained Model Locally

We save the best model and tokenizer to disk for reuse in production or future experiments.

In [ ]:
best_model_dir = "finbert-sentiment-model"

best_trainer.model.save_pretrained(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

best_model_dir

## 13. Sample Predictions on Custom Headlines

We run inference on a few custom financial headlines and show predicted labels with probabilities.

In [ ]:
sample_headlines = [
    "Company posts record quarterly profit amid strong demand",
    "Shares plunge after unexpected regulatory investigation",
    "Firm announces stable outlook and maintains guidance",
    "Bank cuts rates as recession fears grow",
]

best_model = best_trainer.model.to(DEVICE)
best_model.eval()

encoded = tokenizer(
    sample_headlines,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    outputs = best_model(**encoded)
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()

pred_ids = probs.argmax(axis=1)
pred_labels = [id2label[i] for i in pred_ids]

pred_df = pd.DataFrame(
    {
        "headline": sample_headlines,
        "predicted_label": pred_labels,
        "prob_negative": probs[:, 0],
        "prob_neutral": probs[:, 1],
        "prob_positive": probs[:, 2],
    }
)

pred_df

## 14. Real-World Applications in Finance & Stock Market Sentiment

Model outputs can power practical workflows such as:
- **Alerting**: detect sudden negative sentiment spikes on a company.
- **Monitoring**: track sentiment trends over time for risk oversight.
- **Backtesting**: use sentiment signals as features in trading strategies.

Below is a small demo snippet that converts predictions into a simple sentiment score for monitoring.

In [ ]:
demo_texts = df["clean_text"].head(200).tolist()

encoded_demo = tokenizer(
    demo_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
).to(DEVICE)

with torch.no_grad():
    demo_outputs = best_model(**encoded_demo)
    demo_probs = torch.softmax(demo_outputs.logits, dim=1).cpu().numpy()

# Simple sentiment score: positive prob minus negative prob
sentiment_scores = demo_probs[:, 2] - demo_probs[:, 0]

score_summary = {
    "mean_score": float(np.mean(sentiment_scores)),
    "min_score": float(np.min(sentiment_scores)),
    "max_score": float(np.max(sentiment_scores)),
}

score_summary